# Loops, interactively: discovering the invariant

Straight-line code proved itself in notebook 01 (`py_prove` just executes
symbolically). A `while` loop is different: you must supply the two facts
symbolic execution cannot find — a **loop invariant** (what stays true at
the loop head) and a **decreasing measure** (why it terminates). Everything
else — the logical state, the environment rendering, the body's effect — is
derived by `py_begin` / `py_loop` (`LeanModels/Python/LoopTactic.lean`).

This notebook is the honest workflow: **run the program → tabulate the loop
head → guess an invariant → get stuck → read the residual goal → fix it.**

Run `lake build` at the repo root once before starting.

In [1]:
import pathlib, sys
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "lakefile.toml").exists())
sys.path.insert(0, str(root / "tools"))
%load_ext lean_magic

The function under study — the triangular-number loop (same as
`Examples/tri/tri.py`):

In [2]:
%%pyfile tri.py
def tri(n):
    total = 0
    i = 0
    while i <= n:
        total += i
        i += 1
    return total

## Build the table by hand

An invariant is an empirical fact about the running program before it is a
proof artifact. So run the program:

In [3]:
%pyrun tri(0)
%pyrun tri(1)
%pyrun tri(2)
%pyrun tri(3)
%pyrun tri(10)

Now trace `tri(3)` and write down the state **each time control reaches the
loop head** (just before `while i <= n` is tested):

| loop head # | `total` | `i` | `2*total` | `i*(i-1)` | `i*(i+1)` |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 | 0 |
| 1 | 0 | 1 | 0 | 0 | 2 |
| 2 | 1 | 2 | 2 | 2 | 6 |
| 3 | 3 | 3 | 6 | 6 | 12 |
| 4 | 6 | 4 | 12 | 12 | 20 |

At head #4 the test `4 <= 3` fails and the loop exits with `total = 6`.

Two candidate invariants suggest themselves for "`total` is a triangular
number of `i`" (stated multiplication-free, the way `grind` likes it):

* `2*total = i*(i+1)` — "total is the sum up to **and including** `i`"
* `2*total = i*(i-1)` — "total is the sum up to `i - 1`"

The table already votes for the second (compare the columns!). But suppose
you reached for the first — it's the natural wrong guess, since the body
*does* add `i` to `total`. Let's try it and read the failure.

## The wrong invariant, and how to read the stuck goals

The cell below stops right after `py_loop` — no closing tactics — so the
**residual goals** print in full. They are pure arithmetic: no `Val`, no
fuel, no AST. Four goals, in order:

1. **exit algebra** — primed variables (`total'`, `i'`) are the loop-exit
   values: from the invariant plus `hcont : ¬ i' ≤ n`, conclude the returned
   value equals the spec;
2. **`case hinv` — invariant preservation** — one body step: invariant at
   (`total`, `i`) plus the loop test implies the invariant at
   (`total + i`, `i + 1`);
3. **`case hdec` — measure decrease**;
4. **`case hinit` — invariant at entry** (`total = 0`, `i = 0`).

In [4]:
%%lean tri_wrong_invariant
theorem tri_total (n : PyInt) (hn : 0 ≤ n) : tri(n) ==> n * (n + 1) / 2 := by
  py_begin [tri]
  py_loop (inv := fun (total i : Int) => 2 * total = i * (i + 1))
          (dec := fun (total i : Int) => (n + 1 - i).toNat)

LeanCellError: Lean reported 1 error(s) in tri_wrong_invariant — see the goal state above

Read `case hinv`, the preservation goal:

```
hinv : 2 * total = i * (i + 1)
hcont : i ≤ n
⊢ 2 * (total + i) = (i + 1) * (i + 1 + 1)
```

This is **falsifiable by instantiation** — plug in the table's head #0
(`total = 0`, `i = 0`): the hypothesis holds (`0 = 0`) but the conclusion is
`0 = 2`. No tactic will close a false goal; the invariant itself is wrong.
The mistake: `total += i` runs *before* `i += 1`, so at the loop head
`total` covers only `0 + 1 + ⋯ + (i-1)`.

The first goal tells the same story from the exit end: from
`2 * total' = i' * (i' + 1)` and `¬ i' ≤ n` you'd conclude
`total' = (n+1)(n+2)/2` — one triangle too many.

**The fix**, straight from the table: `2*total = i*(i-1)`. One more
ingredient: the exit algebra needs to pin `i'` exactly. `¬ i' ≤ n` only
gives `i' > n`, so the invariant must also carry the **range**
`0 ≤ i ∧ i ≤ n + 1` — then at exit `i' = n + 1` and the division falls out.
Forgetting the range is the *other* classic stuck goal: everything closes
except the first bullet.

In [5]:
%%lean tri_proof
#py_check tri(10) = 55
#py_check tri(0) = 0

theorem tri_total (n : PyInt) (hn : 0 ≤ n) : tri(n) ==> n * (n + 1) / 2 := by
  py_begin [tri]
  py_loop (inv := fun (total i : Int) => 0 ≤ i ∧ i ≤ n + 1 ∧ 2 * total = i * (i - 1))
          (dec := fun (total i : Int) => (n + 1 - i).toNat)
  · obtain rfl : i' = n + 1 := by omega
    grind
  all_goals grind

Anatomy of the proof, clause by clause:

* `py_begin [tri]` — symbolically executes the entry (`total = 0`, `i = 0`)
  up to the loop.
* `py_loop (inv := …) (dec := …)` — the **lambda binder names are matched
  against the Python variable names** (`total`, `i`); from just these two
  clauses the tactic derives the logical state and applies the generic
  while rule. (If a theorem binder shadows a mutated Python variable —
  e.g. a count-*down* loop consuming `n` — there's a
  `(state := [...])` escape hatch; see `Examples/sum_to/sum_to.py`.)
* the bullet `·` handles the exit-algebra goal: `omega` pins `i' = n + 1`,
  `grind` finishes the division; `all_goals grind` sweeps preservation,
  measure, and initialization.

Same statement, same invariant content as the pure-Lean proof of
`n*(n+1)/2` — the loop machinery added no proof burden of its own.

## Your turn: `count_up`

The smallest possible loop — count from `0` up to `n`:

In [6]:
%%pyfile count_up.py
def count_up(n):
    c = 0
    while c < n:
        c += 1
    return c

In [7]:
%pyrun count_up(0)
%pyrun count_up(5)
%pyrun count_up(42)

Prove `count_up(n) ==> n` for `0 ≤ n`. Fill in the scaffold below (replace
`sorry`; the cell runs as-is, with a warning). Hints, in escalating order:

1. One loop variable ⇒ one lambda binder: `fun (c : Int) => …`.
2. What is true about `c` at *every* loop head? (Two inequalities; there is
   no sum to track this time.)
3. Exit gives `hcont : ¬ c' < n`. Which invariant conjunct turns that into
   `c' = n`?
4. The measure is the distance still to climb, as a `Nat` (`.toNat`).

If you get stuck, do what this notebook did: leave off the closing tactics
and *read the residual goals*.

In [8]:
%%lean count_up_exercise
theorem count_up_total (n : PyInt) (hn : 0 ≤ n) : count_up(n) ==> n := by
  -- Your turn. The shape mirrors tri_proof:
  --   py_begin [count_up]
  --   py_loop (inv := fun (c : Int) => _)
  --           (dec := fun (c : Int) => _)
  --   all_goals grind
  sorry

---

### Solution

(scroll only when done)

<br><br><br><br><br><br><br><br>

In [9]:
%%lean count_up_solution
#py_check count_up(5) = 5

theorem count_up_total (n : PyInt) (hn : 0 ≤ n) : count_up(n) ==> n := by
  py_begin [count_up]
  py_loop (inv := fun (c : Int) => 0 ≤ c ∧ c ≤ n)
          (dec := fun (c : Int) => (n - c).toNat)
  all_goals grind

`0 ≤ c ∧ c ≤ n` is all of it: at exit, `¬ c' < n` plus `c' ≤ n` forces
`c' = n` — the invariant's upper bound is exactly the fact that turns the
negated test into the answer. The measure `(n - c).toNat` decreases because
the test guarantees `c < n` inside the body.

## Where to next

* `Examples/sum_to/sum_to.py` (the inline `# lean[` showcase) and
  `Examples/gcd/proof.lean` — count-down loops, where the
  mutated Python variable shadows a theorem binder and the invariant needs
  the initial value: the `(state := [...])` escape hatch.
* `Examples/tri/proof.lean` — this notebook's proof in house style, plus the
  `@[spec]` corollary forms (`py_corollary`) that downstream proofs consume.
* `docs/spec-surface.md` — partial vs. total (`~~>` vs `==>`), exceptions
  as postconditions (`==>!`), relational specs (`⇓`).